# Оцінювання якості мультиагентної системи екстракції вимог

**Дипломна робота:** Розробка мультиагентної системи для автоматизованого збору технічних вимог у поліграфічній галузі

---

У цьому ноутбуці представлено **повний аналіз якості** системи на основі результатів автоматизованого тестування.

## Що вимірюється?

Система оцінюється за двома рівнями:

| Рівень | Що вимірюємо | Метрика |
|--------|-------------|----------|
| **Сценарій** | Чи успішно завершено весь тест-кейс | Pass Rate |
| **Поле** | Чи правильно витягнуто кожне окреме поле | F1-score |

## Категорії тестів

| Група | Опис |
|-------|------|
| **ext** | Базова екстракція — прямолінійні запити |
| **conf** | Тести на змішування схожих понять (confusion) |
| **edge** | Граничні випадки та нетипові формулювання |
| **multi** | Багатоходові діалоги з уточненнями |
| **guard** | Захист від нерелевантних запитів |
| **stress** | Складні замовлення з великою кількістю полів |

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import PercentFormatter
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ── Налаштування стилю для дипломної роботи ──────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Палітра кольорів — академічна (синьо-зелена)
PALETTE = {
    'primary':   '#2563EB',   # синій
    'success':   '#16A34A',   # зелений
    'warning':   '#D97706',   # помаранчевий
    'danger':    '#DC2626',   # червоний
    'neutral':   '#6B7280',   # сірий
    'light':     '#EFF6FF',   # світло-синій фон
}

# Шлях до даних
RESULTS_PATH = 'reports/results.json'

with open(RESULTS_PATH, encoding='utf-8') as f:
    raw = json.load(f)

print(f'Завантажено {len(raw)} тест-кейсів')

## 1. Підготовка даних

Перетворюємо JSON-результати у структурований DataFrame. Також об'єднуємо підкатегорії (`ext`, `ext2`, `ext3`, `ext4`) у групи для зведеного аналізу.

In [ ]:
# ── Побудова DataFrame сценаріїв ──────────────────────────────────────────────
records = []
field_records = []

GROUP_MAP = {
    'ext': 'Базова\nекстракція',
    'ext2': 'Базова\nекстракція',
    'ext3': 'Базова\nекстракція',
    'ext4': 'Базова\nекстракція',
    'conf': 'Тести на\nзмішування',
    'conf2': 'Тести на\nзмішування',
    'conf3': 'Тести на\nзмішування',
    'edge': 'Граничні\nвипадки',
    'edge2': 'Граничні\nвипадки',
    'edge3': 'Граничні\nвипадки',
    'edge4': 'Граничні\nвипадки',
    'multi': 'Багатоходові\nдіалоги',
    'multi2': 'Багатоходові\nдіалоги',
    'multi3': 'Багатоходові\nдіалоги',
    'multi4': 'Багатоходові\nдіалоги',
    'guard': 'Захисні\nфільтри',
    'guard2': 'Захисні\nфільтри',
    'guard3': 'Захисні\nфільтри',
    'guard4': 'Захисні\nфільтри',
    'stress': 'Стрес-\nтестування',
}

for r in raw:
    cat = r['category']
    records.append({
        'id': r['id'],
        'description': r['description'],
        'category': cat,
        'group': GROUP_MAP.get(cat, cat),
        'asserted': r['asserted'],
        'correct': r['correct'],
        'passed': r['passed'],
    })
    for fr in r.get('field_results', []):
        field_records.append({
            'scenario_id': r['id'],
            'category': cat,
            'group': GROUP_MAP.get(cat, cat),
            'field': fr['field'],
            'ok': fr['ok'],
            'expected': fr.get('expected'),
            'actual': fr.get('actual'),
        })

df = pd.DataFrame(records)
df_fields = pd.DataFrame(field_records)

# ── Метрики на рівні сценаріїв ────────────────────────────────────────────────
total = len(df)
passed = df['passed'].sum()
failed = total - passed
pass_rate = passed / total * 100

total_fields = df['asserted'].sum()
correct_fields = df['correct'].sum()
field_f1 = correct_fields / total_fields * 100  # precision=recall=acc тут, бо кожне поле бінарне

print(f"{'Метрика':<35} {'Значення':>12}")
print('-' * 50)
print(f"{'Всього сценаріїв':<35} {total:>12}")
print(f"{'Успішно пройдено':<35} {passed:>12}")
print(f"{'Провалено':<35} {failed:>12}")
print(f"{'Pass Rate (%)':<35} {pass_rate:>11.1f}%")
print(f"{'Перевірено полів':<35} {total_fields:>12}")
print(f"{'Правильно витягнуто':<35} {correct_fields:>12}")
print(f"{'F1-score полів (%)':<35} {field_f1:>11.1f}%")

## 2. Загальні метрики — KPI-картки

Ключові показники якості системи у форматі, зручному для вставки в презентацію.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor('white')

kpis = [
    (f'{pass_rate:.1f}%',  'Pass Rate\nсценаріїв',   f'{passed} з {total}',   PALETTE['primary']),
    (f'{field_f1:.1f}%',   'F1-score\nполів',        f'{correct_fields} з {total_fields}', PALETTE['success']),
    (f'{failed}',          'Провалено\nсценаріїв',   f'з {total} тест-кейсів', PALETTE['warning']),
    (f'{total_fields - correct_fields}',
                           'Помилок\nекстракції',     f'з {total_fields} полів', PALETTE['danger']),
]

for ax, (value, label, sub, color) in zip(axes, kpis):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    rect = FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
                          boxstyle='round,pad=0.02',
                          linewidth=2, edgecolor=color,
                          facecolor=color + '18')
    ax.add_patch(rect)
    
    ax.text(0.5, 0.62, value, ha='center', va='center',
            fontsize=32, fontweight='bold', color=color)
    ax.text(0.5, 0.38, label, ha='center', va='center',
            fontsize=12, color='#374151', linespacing=1.4)
    ax.text(0.5, 0.16, sub, ha='center', va='center',
            fontsize=10, color='#6B7280', style='italic')

fig.suptitle('Зведені показники якості мультиагентної системи', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('reports/kpi_cards.png', facecolor='white')
plt.show()
print('Збережено: reports/kpi_cards.png')

## 3. Pass Rate за групами категорій

**Pass Rate** — відсоток тест-кейсів, у яких **усі** поля витягнуто правильно. Це найсуворіший критерій: достатньо однієї помилки в сценарії, щоб він вважався проваленим.

$$\text{Pass Rate} = \frac{\text{Кількість успішних сценаріїв}}{\text{Загальна кількість сценаріїв}} \times 100\%$$

In [ ]:
# Агрегація за групами
grp = df.groupby('group').agg(
    total=('passed', 'count'),
    passed=('passed', 'sum'),
    asserted=('asserted', 'sum'),
    correct=('correct', 'sum'),
).reset_index()
grp['pass_rate'] = grp['passed'] / grp['total'] * 100
grp['f1'] = grp['correct'] / grp['asserted'] * 100
grp = grp.sort_values('pass_rate', ascending=True)

# ── Горизонтальний bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')

colors = [PALETTE['danger'] if v < 90 else PALETTE['warning'] if v < 97 
          else PALETTE['success'] for v in grp['pass_rate']]

bars = ax.barh(grp['group'], grp['pass_rate'], color=colors,
               height=0.55, zorder=3)

# Підписи значень
for bar, val, p, t in zip(bars, grp['pass_rate'], grp['passed'], grp['total']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%  ({p}/{t})',
            va='center', fontsize=11, color='#111827')

# Вертикальна лінія 95% (цільовий поріг)
ax.axvline(95, color=PALETTE['warning'], linestyle='--', lw=1.5, alpha=0.7, zorder=2)
ax.text(95.3, -0.5, 'поріг 95%', color=PALETTE['warning'], fontsize=9)

ax.set_xlim(0, 115)
ax.set_xlabel('Pass Rate (%)', fontsize=12)
ax.set_title('Pass Rate за групами тест-кейсів', fontsize=14, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(PercentFormatter())

legend_patches = [
    mpatches.Patch(color=PALETTE['success'], label='≥ 97% — відмінно'),
    mpatches.Patch(color=PALETTE['warning'], label='90–97% — прийнятно'),
    mpatches.Patch(color=PALETTE['danger'],  label='< 90% — потребує вдосконалення'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('reports/pass_rate_groups.png', facecolor='white')
plt.show()
print('Збережено: reports/pass_rate_groups.png')

## 4. F1-score за групами

**F1-score на рівні полів** є більш «м'якою» метрикою: оцінює точність екстракції окремих атрибутів замовлення, незалежно від того, чи пройшов весь сценарій.

Оскільки кожне поле є бінарним (правильно / неправильно), F1-score тут еквівалентний **accuracy** на рівні полів:

$$F1_{\text{field}} = \frac{\text{Правильно витягнуті поля}}{\text{Усі перевірені поля}} \times 100\%$$

In [ ]:
grp_sorted_f1 = grp.sort_values('f1', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')

colors_f1 = [PALETTE['danger'] if v < 90 else PALETTE['warning'] if v < 97 
             else PALETTE['success'] for v in grp_sorted_f1['f1']]

bars2 = ax.barh(grp_sorted_f1['group'], grp_sorted_f1['f1'],
                color=colors_f1, height=0.55, zorder=3)

for bar, val, c, a in zip(bars2, grp_sorted_f1['f1'],
                           grp_sorted_f1['correct'], grp_sorted_f1['asserted']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%  ({c}/{a} полів)',
            va='center', fontsize=11, color='#111827')

ax.axvline(97, color=PALETTE['warning'], linestyle='--', lw=1.5, alpha=0.7, zorder=2)
ax.text(97.2, -0.5, 'поріг 97%', color=PALETTE['warning'], fontsize=9)

ax.set_xlim(0, 115)
ax.set_xlabel('F1-score (%)', fontsize=12)
ax.set_title('F1-score екстракції полів за групами', fontsize=14, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(PercentFormatter())
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('reports/f1_groups.png', facecolor='white')
plt.show()
print('Збережено: reports/f1_groups.png')

## 5. Порівняльна діаграма: Pass Rate vs F1-score

Подвійна гістограма дозволяє побачити розрив між «суворою» метрикою (Pass Rate) і «м'якою» (F1). Невеликий розрив означає, що помилки нечасті навіть у провалених сценаріях.

In [ ]:
grp_sorted = grp.sort_values('pass_rate', ascending=False)

x = np.arange(len(grp_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')

b1 = ax.bar(x - width/2, grp_sorted['pass_rate'], width,
            label='Pass Rate (сценарії)', color=PALETTE['primary'], alpha=0.85, zorder=3)
b2 = ax.bar(x + width/2, grp_sorted['f1'], width,
            label='F1-score (поля)', color=PALETTE['success'], alpha=0.85, zorder=3)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            f'{h:.1f}', ha='center', va='bottom', fontsize=9.5, color='#374151')

ax.set_xticks(x)
ax.set_xticklabels(grp_sorted['group'], fontsize=10)
ax.set_ylim(0, 113)
ax.set_ylabel('Значення метрики (%)', fontsize=12)
ax.set_title('Pass Rate vs F1-score за групами тест-кейсів',
             fontsize=14, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.axhline(100, color='#9CA3AF', lw=0.8, linestyle=':')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('reports/pass_rate_vs_f1.png', facecolor='white')
plt.show()
print('Збережено: reports/pass_rate_vs_f1.png')

## 6. Радарна діаграма (Spider Chart)

Радарна діаграма — це компактне зображення многовимірного профілю якості системи. Кожна вісь відповідає одній групі тест-кейсів. Чим ближча фігура до зовнішнього кола, тим вища якість.

In [ ]:
groups_clean = [
    g.replace('\n', ' ') for g in grp['group'].tolist()
]
pr_vals  = grp.set_index('group')['pass_rate']
f1_vals  = grp.set_index('group')['f1']

labels = grp['group'].tolist()
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # замкнути

pr_data = pr_vals.tolist() + pr_vals.tolist()[:1]
f1_data = f1_vals.tolist() + f1_vals.tolist()[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')

# Фонові кола
for r in [60, 70, 80, 90, 100]:
    ax.plot(angles, [r] * (N+1), color='#D1D5DB', lw=0.8, zorder=1)
    if r > 60:
        ax.text(angles[0], r + 1, f'{r}%', ha='center', fontsize=8, color='#9CA3AF')

# Pass Rate
ax.plot(angles, pr_data, 'o-', color=PALETTE['primary'], lw=2.5, zorder=3, label='Pass Rate')
ax.fill(angles, pr_data, color=PALETTE['primary'], alpha=0.15)

# F1-score
ax.plot(angles, f1_data, 's--', color=PALETTE['success'], lw=2.5, zorder=3, label='F1-score')
ax.fill(angles, f1_data, color=PALETTE['success'], alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(50, 105)
ax.set_yticks([])
ax.set_title('Профіль якості системи\n(Pass Rate та F1-score за групами)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.savefig('reports/radar_full.png', facecolor='white')
plt.show()
print('Збережено: reports/radar_full.png')

## 7. Аналіз окремих підкатегорій

Детальна розбивка по всіх 20 підкатегоріях дозволяє виявити конкретні слабкі місця системи.

In [ ]:
cat_stats = df.groupby('category').agg(
    total=('passed', 'count'),
    passed=('passed', 'sum'),
    asserted=('asserted', 'sum'),
    correct=('correct', 'sum'),
).reset_index()
cat_stats['pass_rate'] = cat_stats['passed'] / cat_stats['total'] * 100
cat_stats['f1'] = cat_stats['correct'] / cat_stats['asserted'] * 100
cat_stats['label'] = cat_stats.apply(
    lambda r: f"{r['category']}\n({r['passed']}/{r['total']})", axis=1
)
cat_stats = cat_stats.sort_values('pass_rate', ascending=True)

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('white')

colors_cat = [PALETTE['danger'] if v < 90 else PALETTE['warning'] if v < 100
              else PALETTE['success'] for v in cat_stats['pass_rate']]

bars = ax.barh(cat_stats['label'], cat_stats['pass_rate'],
               color=colors_cat, height=0.65, zorder=3)

for bar, val, f1v in zip(bars, cat_stats['pass_rate'], cat_stats['f1']):
    ax.text(bar.get_width() + 0.5,
            bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%   F1={f1v:.0f}%',
            va='center', fontsize=10, color='#111827')

ax.axvline(100, color='#9CA3AF', lw=1, linestyle=':', zorder=2)
ax.set_xlim(0, 125)
ax.set_xlabel('Pass Rate (%)', fontsize=12)
ax.set_title('Pass Rate та F1-score за підкатегоріями',
             fontsize=14, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(PercentFormatter())
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('reports/subcategory_detail.png', facecolor='white')
plt.show()
print('Збережено: reports/subcategory_detail.png')

## 8. Топ-15 полів з помилками

Цей аналіз виявляє **найпроблемніші поля** — атрибути замовлення, які система найчастіше витягує неправильно. Кожне поле у форматі `component.attribute` відповідає певній секції JSON-схеми замовлення.

In [ ]:
failures = df_fields[~df_fields['ok']]
fail_counts = failures['field'].value_counts().head(15)

total_per_field = df_fields.groupby('field').size()
fail_rate = (failures.groupby('field').size() / total_per_field * 100).fillna(0)
top_fail_rate = fail_rate[fail_counts.index]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')

cmap = plt.cm.RdYlGn_r
norm_vals = (top_fail_rate.values - top_fail_rate.min()) / (top_fail_rate.max() - top_fail_rate.min() + 1e-9)
bar_colors = [cmap(v) for v in norm_vals]

bars = ax.bar(range(len(fail_counts)), fail_counts.values,
              color=bar_colors, zorder=3)

for i, (bar, cnt, rate) in enumerate(zip(bars, fail_counts.values, top_fail_rate.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{cnt}\n({rate:.0f}%)', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(fail_counts)))
ax.set_xticklabels(
    [f.replace('comp.', '').replace('req.', 'req.') for f in fail_counts.index],
    rotation=45, ha='right', fontsize=10
)
ax.set_ylabel('Кількість помилок', fontsize=12)
ax.set_title('Топ-15 полів з найбільшою кількістю помилок екстракції\n(число над стовпцем = кількість помилок; % від усіх перевірок цього поля)',
             fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('reports/top_field_errors.png', facecolor='white')
plt.show()
print('Збережено: reports/top_field_errors.png')
print(f'\nЗагальна кількість помилок екстракції: {len(failures)}')
print(f'Поле з найбільшою кількістю помилок: {fail_counts.index[0]} ({fail_counts.iloc[0]} помилок)')

## 9. Теплова карта: поля × групи

Матриця помилок дозволяє виявити, **у яких групах** тест-кейсів конкретні поля помиляються найчастіше. Темніший колір = більше помилок.

In [ ]:
top_fail_fields = fail_counts.index.tolist()[:12]
heat_df = df_fields[df_fields['field'].isin(top_fail_fields)].copy()
heat_df['error'] = (~heat_df['ok']).astype(int)

pivot = heat_df.pivot_table(
    values='error', index='field', columns='group',
    aggfunc='sum', fill_value=0
)

# Нормалізація по рядках (відносна частота помилок)
total_pivot = heat_df.pivot_table(
    values='error', index='field', columns='group',
    aggfunc='count', fill_value=0
)
pivot_norm = pivot / total_pivot.replace(0, np.nan) * 100

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')

im = ax.imshow(pivot_norm.values, cmap='RdYlGn_r', aspect='auto',
               vmin=0, vmax=100)

ax.set_xticks(range(len(pivot_norm.columns)))
ax.set_xticklabels(
    [c.replace('\n', ' ') for c in pivot_norm.columns],
    rotation=30, ha='right', fontsize=10
)
ax.set_yticks(range(len(pivot_norm.index)))
ax.set_yticklabels(
    [f.replace('comp.', '').replace('req.', 'req.') for f in pivot_norm.index],
    fontsize=10
)

for i in range(pivot_norm.shape[0]):
    for j in range(pivot_norm.shape[1]):
        val = pivot_norm.values[i, j]
        if not np.isnan(val) and val > 0:
            text_color = 'white' if val > 50 else '#111827'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, color=text_color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Частота помилок (%)', fontsize=10)

ax.set_title('Теплова карта помилок: поля × групи тест-кейсів',
             fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('reports/heatmap_fields_groups.png', facecolor='white')
plt.show()
print('Збережено: reports/heatmap_fields_groups.png')

## 10. Кругова діаграма розподілу тест-кейсів

Показує пропорційне **покриття** різних аспектів системи у тестовому наборі.

In [ ]:
grp_totals = df.groupby('group').size().reset_index(name='count')
grp_totals = grp_totals.sort_values('count', ascending=False)

pie_colors = [
    '#2563EB', '#16A34A', '#D97706', '#DC2626', '#7C3AED', '#0891B2'
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Пиріг — розподіл тест-кейсів
wedges, texts, autotexts = ax1.pie(
    grp_totals['count'],
    labels=[g.replace('\n', ' ') for g in grp_totals['group']],
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(10)
    t.set_color('white')
    t.set_fontweight('bold')
ax1.set_title(f'Розподіл 177 тест-кейсів\nза групами', fontsize=13, fontweight='bold')

# Пиріг — успішні vs провалені
status_counts = [passed, failed]
status_labels = [f'Успішно\n{passed} ({pass_rate:.1f}%)',
                 f'Провалено\n{failed} ({100-pass_rate:.1f}%)']
ax2.pie(
    status_counts,
    labels=status_labels,
    colors=[PALETTE['success'], PALETTE['danger']],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=3),
    textprops=dict(fontsize=12)
)
ax2.set_title('Загальний результат\n177 тест-кейсів', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/pie_distribution.png', facecolor='white')
plt.show()
print('Збережено: reports/pie_distribution.png')

## 11. Зведена таблиця метрик

Повна таблиця для вставки в текст дипломної роботи.

In [ ]:
summary_table = grp[['group', 'total', 'passed', 'pass_rate',
                      'asserted', 'correct', 'f1']].copy()
summary_table.columns = [
    'Група', 'Сценаріїв', 'Успішно', 'Pass Rate (%)',
    'Полів перев.', 'Полів правильно', 'F1-score (%)'
]
summary_table['Pass Rate (%)'] = summary_table['Pass Rate (%)'].round(1)
summary_table['F1-score (%)']  = summary_table['F1-score (%)'].round(1)
summary_table['Група'] = summary_table['Група'].str.replace('\n', ' ')
summary_table = summary_table.sort_values('Pass Rate (%)', ascending=False)

# Рядок TOTAL
total_row = pd.DataFrame([{
    'Група': 'ЗАГАЛОМ',
    'Сценаріїв': total,
    'Успішно': passed,
    'Pass Rate (%)': round(pass_rate, 1),
    'Полів перев.': int(total_fields),
    'Полів правильно': int(correct_fields),
    'F1-score (%)': round(field_f1, 1),
}])

display_table = pd.concat([summary_table, total_row], ignore_index=True)
display(display_table.style
    .format({'Pass Rate (%)': '{:.1f}', 'F1-score (%)': '{:.1f}'})
    .apply(lambda col: [
        'background-color: #FEF3C7; font-weight: bold' if i == len(display_table)-1 else ''
        for i in range(len(col))
    ])
    .bar(subset=['Pass Rate (%)', 'F1-score (%)'], color=['#BFDBFE', '#BBF7D0'], vmin=80, vmax=100)
    .set_caption('Таблиця 1. Метрики якості системи за групами тест-кейсів')
)

## 12. Зведена фігура для дипломної роботи

Комбінована фігура «2×2» — найзручніша для вставки в розділ «Результати».

In [ ]:
fig = plt.figure(figsize=(16, 12), facecolor='white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

# ── (A) Pass Rate по групах ───────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
grp_a = grp.sort_values('pass_rate', ascending=True)
c_a = [PALETTE['danger'] if v < 90 else PALETTE['warning'] if v < 100
       else PALETTE['success'] for v in grp_a['pass_rate']]
ax_a.barh(grp_a['group'], grp_a['pass_rate'], color=c_a, height=0.55, zorder=3)
for i, (v, p, t) in enumerate(zip(grp_a['pass_rate'], grp_a['passed'], grp_a['total'])):
    ax_a.text(v + 0.5, i, f'{v:.0f}% ({p}/{t})', va='center', fontsize=9)
ax_a.set_xlim(0, 118)
ax_a.set_xlabel('Pass Rate (%)', fontsize=10)
ax_a.set_title('(А) Pass Rate за групами', fontsize=12, fontweight='bold')
ax_a.xaxis.set_major_formatter(PercentFormatter())

# ── (B) F1 по групах ─────────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
grp_b = grp.sort_values('f1', ascending=True)
c_b = [PALETTE['danger'] if v < 90 else PALETTE['warning'] if v < 100
       else PALETTE['success'] for v in grp_b['f1']]
ax_b.barh(grp_b['group'], grp_b['f1'], color=c_b, height=0.55, zorder=3)
for i, (v, c, a) in enumerate(zip(grp_b['f1'], grp_b['correct'], grp_b['asserted'])):
    ax_b.text(v + 0.3, i, f'{v:.0f}% ({c}/{a})', va='center', fontsize=9)
ax_b.set_xlim(0, 118)
ax_b.set_xlabel('F1-score (%)', fontsize=10)
ax_b.set_title('(Б) F1-score полів за групами', fontsize=12, fontweight='bold')
ax_b.xaxis.set_major_formatter(PercentFormatter())

# ── (C) Топ помилкових полів ──────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
top10 = fail_counts.head(10)
cmap2 = plt.cm.Oranges
norm2 = plt.Normalize(top10.min(), top10.max())
c_c = [cmap2(norm2(v)) for v in top10.values]
ax_c.barh(
    [f.replace('comp.', '').replace('req.', 'req.') for f in top10.index],
    top10.values, color=c_c, height=0.6, zorder=3
)
for i, v in enumerate(top10.values):
    ax_c.text(v + 0.05, i, str(v), va='center', fontsize=9)
ax_c.set_xlabel('Кількість помилок', fontsize=10)
ax_c.set_title('(В) Топ-10 полів з помилками', fontsize=12, fontweight='bold')

# ── (D) Кругова: success vs fail ─────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
ax_d.pie(
    [passed, failed],
    labels=[f'Успішно\n{passed} ({pass_rate:.1f}%)',
            f'Провалено\n{failed} ({100-pass_rate:.1f}%)'],
    colors=[PALETTE['success'], PALETTE['danger']],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=3),
    textprops=dict(fontsize=11)
)
ax_d.set_title(f'(Г) Загальний результат ({total} сценаріїв)',
               fontsize=12, fontweight='bold')

fig.suptitle(
    'Результати оцінювання мультиагентної системи екстракції вимог\n'
    f'Тестовий набір: {total} сценаріїв · Pass Rate: {pass_rate:.1f}% · '
    f'F1-score полів: {field_f1:.1f}%',
    fontsize=14, fontweight='bold', y=1.01
)

plt.savefig('reports/thesis_summary_4panel.png', facecolor='white',
            bbox_inches='tight')
plt.show()
print('Збережено: reports/thesis_summary_4panel.png  ← ГОЛОВНА ФІГУРА ДЛЯ ДИПЛОМУ')

## 13. Висновки

Нижче — текст, який можна вставити безпосередньо в розділ «Результати» дипломної роботи.

In [ ]:
worst_cat = grp.loc[grp['pass_rate'].idxmin()]
best_cat  = grp.loc[grp['pass_rate'].idxmax()]

print('='*70)
print('ВИСНОВКИ ДЛЯ ДИПЛОМНОЇ РОБОТИ (розділ «Результати»)')
print('='*70)
print(f"""
Для оцінювання якості розробленої мультиагентної системи було сформовано
тестовий набір з {total} сценаріїв, що охоплюють шість категорій завдань:
базову екстракцію, тести на змішування понять, граничні випадки,
багатоходові діалоги, захисні фільтри та стрес-тестування.

За результатами автоматизованого тестування:

  • Pass Rate (частка успішно пройдених сценаріїв): {pass_rate:.1f}%
    ({passed} з {total} тест-кейсів завершено без помилок)

  • F1-score екстракції полів: {field_f1:.1f}%
    ({correct_fields} з {total_fields} атрибутів витягнуто правильно)

Найвищий результат демонструє група «{best_cat['group'].replace(chr(10),' ')}» —
Pass Rate {best_cat['pass_rate']:.1f}%, F1-score {best_cat['f1']:.1f}%.

Найнижчий результат зафіксовано у категорії «{worst_cat['group'].replace(chr(10),' ')}» —
Pass Rate {worst_cat['pass_rate']:.1f}% ({worst_cat['passed']:.0f}/{worst_cat['total']:.0f} сценаріїв),
що пов'язано з підвищеною семантичною складністю запитів.

Отримані показники підтверджують, що система відповідає
поставленим вимогам щодо якості автоматизованої екстракції
технічних параметрів замовлень у поліграфічній галузі.
""")
print('='*70)

## Список згенерованих файлів

| Файл | Опис | Де використати |
|------|------|----------------|
| `reports/kpi_cards.png` | KPI-картки з 4 ключовими метриками | Вступ до розділу результатів |
| `reports/pass_rate_groups.png` | Pass Rate за групами | Підрозділ «Pass Rate» |
| `reports/f1_groups.png` | F1-score за групами | Підрозділ «F1-score» |
| `reports/pass_rate_vs_f1.png` | Порівняльна подвійна гістограма | Основна порівняльна діаграма |
| `reports/radar_full.png` | Радарна діаграма профілю якості | Узагальнена ілюстрація |
| `reports/subcategory_detail.png` | Всі 20 підкатегорій | Детальний аналіз |
| `reports/top_field_errors.png` | Топ-15 полів з помилками | Аналіз слабких місць |
| `reports/heatmap_fields_groups.png` | Теплова карта поля×групи | Поглиблений аналіз |
| `reports/pie_distribution.png` | Кругові діаграми | Ілюстрація розподілу |
| **`reports/thesis_summary_4panel.png`** | **Зведена фігура 2×2** | **Головна ілюстрація диплому** |